# K-Nearest Neighbors (KNN)

KNN is a **lazy learner** — it stores all training data and classifies new points by a majority vote of their K nearest neighbours in feature space.

**Dataset:** Iris — classify flower species from petal and sepal measurements.

**Algorithm Flow:**  
1. Store all training examples (no model is built)  
2. For each test point, compute distances to all training points  
3. Identify the K nearest neighbours  
4. Predict the majority class among those K neighbours

In [ ]:
import numpy as np
import matplotlib.pyplot as plt
from sklearn.datasets import load_iris
from sklearn.model_selection import train_test_split
from sklearn.preprocessing import StandardScaler
import sys, os
sys.path.insert(0, r"/home/claude/project/2026_Data_Science_and_Machine_Learning/src/rice_ml/supervised_learning")
from knn import KNN, distance
np.random.seed(42)
print("Imports complete")

## Dataset Overview

The **Iris dataset** contains 150 samples of three iris species with four numeric features each.

| Feature | Description |
|---------|-------------|
| Sepal length | Length of sepal in cm |
| Sepal width | Width of sepal in cm |
| Petal length | Length of petal in cm |
| Petal width | Width of petal in cm |

In [ ]:
data = load_iris()
X, y = data.data, data.target
feature_names = data.feature_names
class_names   = data.target_names

print(f"Samples: {X.shape[0]} | Features: {X.shape[1]} | Classes: {len(class_names)}")
print(f"Class distribution: {dict(zip(class_names, [np.sum(y==i) for i in range(3)]))}")

## Exploratory Data Analysis

Before fitting the model, we visualise the data to understand its structure.  
Good separation between classes means KNN should perform well.

In [ ]:
fig, axes = plt.subplots(1, 2, figsize=(13, 5))
colors = ['#e63946', '#457b9d', '#2a9d8f']

# Plot 1: Petal features (best separation)
for cls in range(3):
    mask = y == cls
    axes[0].scatter(X[mask, 2], X[mask, 3], color=colors[cls],
                    label=class_names[cls], alpha=0.75, edgecolors='k', linewidths=0.4)
axes[0].set_xlabel(feature_names[2])
axes[0].set_ylabel(feature_names[3])
axes[0].set_title("Petal Features (Best Separation)")
axes[0].legend()
axes[0].grid(True, linestyle='--', alpha=0.5)

# Plot 2: Sepal features
for cls in range(3):
    mask = y == cls
    axes[1].scatter(X[mask, 0], X[mask, 1], color=colors[cls],
                    label=class_names[cls], alpha=0.75, edgecolors='k', linewidths=0.4)
axes[1].set_xlabel(feature_names[0])
axes[1].set_ylabel(feature_names[1])
axes[1].set_title("Sepal Features (More Overlap)")
axes[1].legend()
axes[1].grid(True, linestyle='--', alpha=0.5)

plt.suptitle("Iris Feature Scatter Plots by Class", fontsize=14, fontweight='bold')
plt.tight_layout()
plt.show()

## Feature Distributions

Histograms help confirm that petal features separate classes more cleanly than sepal features.

In [ ]:
fig, axes = plt.subplots(2, 4, figsize=(16, 7))
for col_idx, name in enumerate(feature_names):
    for cls in range(3):
        axes[0, col_idx].hist(X[y == cls, col_idx], bins=15, alpha=0.6,
                               color=colors[cls], label=class_names[cls], edgecolor='k', linewidth=0.3)
    axes[0, col_idx].set_title(name, fontsize=9)
    axes[0, col_idx].set_xlabel("Value (cm)")
    if col_idx == 0:
        axes[0, col_idx].set_ylabel("Frequency")
        axes[0, col_idx].legend(fontsize=8)
    axes[0, col_idx].grid(True, linestyle='--', alpha=0.4)

# Box plots
for col_idx, name in enumerate(feature_names):
    data_by_class = [X[y == cls, col_idx] for cls in range(3)]
    bp = axes[1, col_idx].boxplot(data_by_class, patch_artist=True,
                                   labels=class_names)
    for patch, color in zip(bp['boxes'], colors):
        patch.set_facecolor(color)
        patch.set_alpha(0.7)
    axes[1, col_idx].set_title(f"{name} — Box Plot", fontsize=9)
    axes[1, col_idx].grid(True, linestyle='--', alpha=0.4)

plt.suptitle("Feature Distributions by Class", fontsize=14, fontweight='bold')
plt.tight_layout()
plt.show()

## Preprocessing & Train/Test Split

KNN uses Euclidean distance — features on different scales would bias results.  
We standardise with `StandardScaler` (zero mean, unit variance) before splitting.

In [ ]:
scaler = StandardScaler()
X_scaled = scaler.fit_transform(X)

X_train, X_test, y_train, y_test = train_test_split(
    X_scaled, y, test_size=0.20, random_state=42, stratify=y)
print(f"Train: {X_train.shape[0]} samples | Test: {X_test.shape[0]} samples")
print(f"Train class distribution: {np.bincount(y_train)}")
print(f"Test  class distribution: {np.bincount(y_test)}")

## Choosing K — Accuracy vs K

We evaluate accuracy on the test set for K = 1…25.  
The **bias-variance trade-off** means very small K overfits (high variance) and very large K underfits (high bias).

In [ ]:
k_values = list(range(1, 26))
accuracies = []

for k in k_values:
    clf = KNN(k=k)
    clf.fit(X_train, y_train)
    preds = clf.predict(X_test)
    acc = np.mean(np.array(preds) == y_test)
    accuracies.append(acc)

best_k = k_values[accuracies.index(max(accuracies))]

plt.figure(figsize=(10, 5))
plt.plot(k_values, accuracies, 'o-', color='steelblue', linewidth=2, markersize=7)
plt.axvline(best_k, color='tomato', linestyle='--', linewidth=2, label=f'Best K = {best_k}  (acc={max(accuracies):.3f})')
plt.fill_between(k_values, 0.9, accuracies, alpha=0.15, color='steelblue')
plt.xlabel("K (number of neighbours)", fontsize=12)
plt.ylabel("Test Accuracy", fontsize=12)
plt.title("KNN Accuracy vs Number of Neighbours", fontsize=14, fontweight='bold')
plt.xticks(k_values)
plt.ylim(0.88, 1.01)
plt.legend(fontsize=11)
plt.grid(True, linestyle='--', alpha=0.5)
plt.tight_layout()
plt.show()
print(f"Best K = {best_k}, Test Accuracy = {max(accuracies):.4f}")

## Train Best Model & Evaluate

Using the optimal K we found, we train the final classifier and evaluate with a full classification report.

In [ ]:
clf = KNN(k=best_k)
clf.fit(X_train, y_train)
y_pred = np.array(clf.predict(X_test))
accuracy = np.mean(y_pred == y_test)
print(f"Test Accuracy (K={best_k}): {accuracy:.4f}")

from sklearn.metrics import confusion_matrix, classification_report
cm = confusion_matrix(y_test, y_pred)
print("\n--- Classification Report ---")
print(classification_report(y_test, y_pred, target_names=class_names))

## Confusion Matrix

Each row represents the **actual** class; each column represents the **predicted** class.  
High values on the diagonal indicate correct classifications.

In [ ]:
fig, axes = plt.subplots(1, 2, figsize=(13, 5))

# Confusion matrix heatmap
im = axes[0].imshow(cm, cmap='Blues')
axes[0].set_xticks(range(3)); axes[0].set_yticks(range(3))
axes[0].set_xticklabels(class_names, rotation=15); axes[0].set_yticklabels(class_names)
axes[0].set_xlabel("Predicted", fontsize=12); axes[0].set_ylabel("Actual", fontsize=12)
axes[0].set_title(f"Confusion Matrix (K={best_k})", fontsize=13, fontweight='bold')
for i in range(3):
    for j in range(3):
        axes[0].text(j, i, cm[i, j], ha='center', va='center', fontsize=16,
                     color='white' if cm[i, j] > cm.max() / 2 else 'black')
plt.colorbar(im, ax=axes[0])

# Per-class accuracy bar chart
per_class_acc = cm.diagonal() / cm.sum(axis=1)
bars = axes[1].bar(class_names, per_class_acc, color=colors, edgecolor='k', alpha=0.85)
for bar, acc in zip(bars, per_class_acc):
    axes[1].text(bar.get_x() + bar.get_width() / 2, bar.get_height() + 0.01,
                 f"{acc:.3f}", ha='center', fontsize=12, fontweight='bold')
axes[1].set_ylim(0, 1.15)
axes[1].set_ylabel("Per-class Accuracy", fontsize=12)
axes[1].set_title("Accuracy per Class", fontsize=13, fontweight='bold')
axes[1].grid(True, axis='y', linestyle='--', alpha=0.5)

plt.suptitle("Model Evaluation", fontsize=14, fontweight='bold')
plt.tight_layout()
plt.show()

## 2D Decision Boundary (Petal Features)

Plotting the decision boundary with only 2 features (petal length and width) gives a clear visual of how KNN partitions feature space. Darker regions indicate class membership.

In [ ]:
# Use only petal features for 2D boundary
X2 = X_scaled[:, 2:]
X2_tr, X2_te, y2_tr, y2_te = train_test_split(X2, y, test_size=0.20, random_state=42, stratify=y)

clf2 = KNN(k=best_k)
clf2.fit(X2_tr, y2_tr)

h = 0.05
x_min, x_max = X2[:, 0].min() - 0.5, X2[:, 0].max() + 0.5
y_min, y_max = X2[:, 1].min() - 0.5, X2[:, 1].max() + 0.5
xx, yy = np.meshgrid(np.arange(x_min, x_max, h), np.arange(y_min, y_max, h))
grid = np.column_stack([xx.ravel(), yy.ravel()])
Z = np.array(clf2.predict(grid)).reshape(xx.shape)

plt.figure(figsize=(9, 6))
plt.contourf(xx, yy, Z, alpha=0.25, cmap='bwr')
plt.contour(xx, yy, Z, colors='k', linewidths=0.6, alpha=0.4)
for cls in range(3):
    mask = y2_te == cls
    plt.scatter(X2_te[mask, 0], X2_te[mask, 1], color=colors[cls],
                label=class_names[cls], edgecolors='k', linewidths=0.6, s=70, zorder=3)
plt.xlabel("Petal Length (standardised)", fontsize=12)
plt.ylabel("Petal Width (standardised)", fontsize=12)
plt.title(f"KNN Decision Boundary (Petal Features, K={best_k})", fontsize=13, fontweight='bold')
plt.legend()
plt.grid(True, linestyle='--', alpha=0.4)
plt.tight_layout()
plt.show()

## Key Takeaways

- KNN is simple, non-parametric, and needs no explicit training step.
- **Small K** → complex, noisy boundary (low bias, high variance — overfitting).
- **Large K** → smoother boundary (high bias, low variance — underfitting).
- Feature scaling is essential — KNN is based on Euclidean distance.
- KNN is O(n) at prediction time — slow for very large datasets.
- The Iris dataset is nearly linearly separable in petal space, so even K=1 performs well.